In [ ]:
# ── Run this FIRST after reconnecting ────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

CUDA available: True
Device: Tesla T4


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────
!pip install -q huggingface_hub google-generativeai transformers

In [ ]:
# ── Cell 5 FINAL CLEAN: Flan-T5 as primary, rule-based fallback ──
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Flan-T5 already loaded — skip if flan_tok/flan_mdl in memory
flan_tok = AutoTokenizer.from_pretrained('google/flan-t5-large')
flan_mdl = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-large').to('cuda')
flan_mdl.eval()

def build_prompt(name, age, risk, xray, nlp_result):
    syms = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    return (
        f'Patient name is {name}, age {age} years. '
        f'Heart disease risk is {risk["label"]} with a score of {risk["score"]:.0%}. '
        f'Chest X-ray shows {xray["label"]} with {xray["confidence"]:.0%} confidence. '
        f'Symptoms reported are {syms}. '
        f'Urgency level is {nlp_result["urgency"]}. '
        f'As an emergency physician, summarize this case in 3 sentences '
        f'and recommend the next clinical action.'
    )

def call_flan(prompt):
    inp = flan_tok(
        prompt, return_tensors='pt',
        truncation=True, max_length=512
    ).to('cuda')
    with torch.no_grad():
        out = flan_mdl.generate(
            **inp,
            max_new_tokens=200,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    return flan_tok.decode(out[0], skip_special_tokens=True).strip()

def rule_based(name, age, risk, xray, nlp_result):
    syms = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    action = {
        'CRITICAL': 'Immediate transfer to emergency resuscitation bay is required.',
        'MODERATE': 'Admit for monitoring, investigation, and treatment.',
        'MILD':     'Discharge with outpatient follow-up within 48 hours.'
    }.get(nlp_result['urgency'], 'Clinical evaluation is recommended.')
    return (
        f'Patient {name}, {age} years, presents with {risk["label"].lower()} cardiovascular '
        f'risk and {xray["label"].lower()} findings on chest imaging. '
        f'Reported symptoms include {syms}, with urgency classified as {nlp_result["urgency"]}. '
        f'{action}'
    )

print('✅ Flan-T5 ready as primary LLM')

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Flan-T5 ready as primary LLM


In [ ]:
# ── Cell 6 FINAL CLEAN: generate_report() ────────────────────────
def generate_report(name, age, risk, xray, nlp_result):
    prompt = build_prompt(name, age, risk, xray, nlp_result)
    try:
        report = call_flan(prompt)
        source = 'Flan-T5 (local GPU)'
    except Exception as e:
        print(f'Flan failed: {e}')
        report = rule_based(name, age, risk, xray, nlp_result)
        source = 'Rule-based fallback'
    print(f'✅ Report via: {source}')
    return {'report': report}

# ── Test ─────────────────────────────────────────────────────────
r1 = generate_report(
    name='Ramesh Kumar', age=55,
    risk={'label': 'HIGH',     'score': 0.78},
    xray={'label': 'PNEUMONIA','confidence': 0.91},
    nlp_result={'symptoms': ['chest pain', 'breathlessness'], 'urgency': 'CRITICAL'}
)
print(f'\nPatient 1:\n{r1["report"]}\n' + '-'*60)

r2 = generate_report(
    name='Sunita Devi', age=42,
    risk={'label': 'MODERATE', 'score': 0.50},
    xray={'label': 'NORMAL',   'confidence': 0.85},
    nlp_result={'symptoms': ['fever', 'vomiting'], 'urgency': 'MODERATE'}
)
print(f'\nPatient 2:\n{r2["report"]}\n' + '-'*60)

r3 = generate_report(
    name='Arjun Singh', age=28,
    risk={'label': 'LOW',      'score': 0.15},
    xray={'label': 'NORMAL',   'confidence': 0.92},
    nlp_result={'symptoms': ['cough'], 'urgency': 'MILD'}
)
print(f'\nPatient 3:\n{r3["report"]}\n' + '-'*60)

✅ Report via: Flan-T5 (local GPU)

Patient 1:
Ramesh Kumar, age 55 years, has chest pain and breathlessness. Urgency level is CRITICAL.
------------------------------------------------------------
✅ Report via: Flan-T5 (local GPU)

Patient 2:
Chest X-ray shows NORMAL with 85% confidence.
------------------------------------------------------------
✅ Report via: Flan-T5 (local GPU)

Patient 3:
Chest X-ray shows NORMAL with 92% Confidence. Urgency level is MILD.
------------------------------------------------------------


In [ ]:
# ── Cell 5 FINAL: OpenAI GPT-3.5 + Flan-T5 fallback ─────────────
!pip install -q openai

In [ ]:
from openai import OpenAI
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── OpenAI client ─────────────────────────────────────────────────
OPENAI_KEY = ''   # ← paste from platform.openai.com/api-keys
openai_client = OpenAI(api_key=OPENAI_KEY)

# ── Flan-T5 fallback (already loaded — skip if in memory) ─────────
flan_tok = AutoTokenizer.from_pretrained('google/flan-t5-large')
flan_mdl = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-large').to('cuda')
flan_mdl.eval()

SYSTEM_PROMPT = (
    'You are an experienced emergency medicine physician. '
    'Write a concise, factual triage summary in exactly 3 sentences. '
    'Do not fabricate any symptoms not listed. '
    'End with a clear recommended clinical action.'
)

def build_prompt(name, age, risk, xray, nlp_result):
    syms = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    return (
        f'Patient: {name}, Age: {age}\n'
        f'ML Risk Prediction : {risk["label"]} (score {risk["score"]:.0%})\n'
        f'X-Ray Classification: {xray["label"]} ({xray["confidence"]:.0%} confidence)\n'
        f'Symptoms Reported   : {syms}\n'
        f'Urgency Level       : {nlp_result["urgency"]}\n\n'
        f'Write a 3-sentence clinical summary and end with recommended action.'
    )

def call_openai(prompt):
    response = openai_client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt}
        ],
        max_tokens=200,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

def call_flan(prompt):
    inp = flan_tok(
        prompt, return_tensors='pt',
        truncation=True, max_length=512
    ).to('cuda')
    with torch.no_grad():
        out = flan_mdl.generate(
            **inp,
            max_new_tokens=200,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    return flan_tok.decode(out[0], skip_special_tokens=True).strip()

def rule_based(name, age, risk, xray, nlp_result):
    syms   = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    action = {
        'CRITICAL': 'Immediate transfer to emergency resuscitation bay is required.',
        'MODERATE': 'Admit for monitoring, investigation, and treatment.',
        'MILD':     'Discharge with outpatient follow-up within 48 hours.'
    }.get(nlp_result['urgency'], 'Clinical evaluation is recommended.')
    return (
        f'Patient {name}, {age} years, presents with {risk["label"].lower()} cardiovascular '
        f'risk and {xray["label"].lower()} findings on chest imaging. '
        f'Reported symptoms include {syms}, with urgency classified as {nlp_result["urgency"]}. '
        f'{action}'
    )

print('✅ OpenAI + Flan-T5 + Rule-based ready')

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ OpenAI + Flan-T5 + Rule-based ready


In [ ]:
# ── Improved build_prompt() — more detailed input ─────────────────
def build_prompt(name, age, risk, xray, nlp_result):
    syms = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    urgency_desc = {
        'CRITICAL': 'life-threatening, requires immediate intervention',
        'MODERATE': 'serious but stable, requires prompt attention',
        'MILD':     'non-urgent, can be managed with routine care'
    }.get(nlp_result['urgency'], nlp_result['urgency'])

    return (
        f'You are a senior emergency medicine physician writing a formal clinical triage report. '
        f'A patient named {name}, {age} years old, has been brought to the emergency department. '
        f'The AI-based risk assessment system has classified the cardiovascular risk as {risk["label"]} '
        f'with a probability score of {risk["score"]:.0%}. '
        f'The chest X-ray analysis has returned a finding of {xray["label"]} '
        f'with a confidence of {xray["confidence"]:.0%}. '
        f'The patient has reported the following symptoms: {syms}. '
        f'The overall triage urgency has been assessed as {nlp_result["urgency"]}, '
        f'which means the condition is {urgency_desc}. '
        f'Based on all the above clinical data, write a detailed 5-sentence medical summary. '
        f'Sentence 1: Describe the patient and their presenting complaints. '
        f'Sentence 2: Summarize the AI risk and X-ray findings. '
        f'Sentence 3: Explain what these findings suggest clinically. '
        f'Sentence 4: State the urgency level and what it means for this patient. '
        f'Sentence 5: Give a specific recommended clinical action for the treating physician.'
    )

# ── Improved call_flan() — forces longer output ───────────────────
def call_flan(prompt):
    inp = flan_tok(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=600
    ).to('cuda')
    with torch.no_grad():
        out = flan_mdl.generate(
            **inp,
            max_new_tokens=300,       # increased from 200
            min_new_tokens=80,        # forces minimum length
            num_beams=5,              # better quality search
            length_penalty=2.0,       # encourages longer output
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3    # avoids repeating phrases
        )
    return flan_tok.decode(out[0], skip_special_tokens=True).strip()

# ── Improved rule_based() — longer paragraph ──────────────────────
def rule_based(name, age, risk, xray, nlp_result):
    syms   = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    action = {
        'CRITICAL': (
            'This patient must be transferred to the emergency resuscitation bay immediately. '
            'IV access, cardiac monitoring, and oxygen supplementation should be initiated without delay. '
            'A senior physician and crash team must be alerted at once.'
        ),
        'MODERATE': (
            'The patient should be admitted to a monitored ward for further investigation. '
            'Blood investigations, ECG, and specialist consultation are recommended. '
            'Regular vitals monitoring every 2 hours is advised until the condition stabilizes.'
        ),
        'MILD': (
            'The patient may be discharged with appropriate medications and advice. '
            'A follow-up appointment with a general physician within 48 hours is recommended. '
            'The patient should be instructed to return immediately if symptoms worsen.'
        )
    }.get(nlp_result['urgency'], 'Clinical evaluation by a physician is recommended.')

    risk_explain = {
        'HIGH':     'indicating a significant likelihood of underlying cardiac pathology',
        'MODERATE': 'suggesting a moderate probability of cardiovascular involvement',
        'LOW':      'suggesting a low probability of acute cardiac disease'
    }.get(risk['label'], '')

    xray_explain = {
        'PNEUMONIA': 'consistent with an active pulmonary infection requiring treatment',
        'NORMAL':    'showing no acute cardiopulmonary abnormality at this time'
    }.get(xray['label'], '')

    return (
        f'Patient {name}, {age} years of age, has presented to the emergency department '
        f'with the following reported symptoms: {syms}. '
        f'The AI-powered risk assessment has classified the cardiovascular risk as {risk["label"]} '
        f'({risk["score"]:.0%} probability score), {risk_explain}. '
        f'Chest X-ray imaging has returned a result of {xray["label"]} '
        f'({xray["confidence"]:.0%} confidence), {xray_explain}. '
        f'The overall triage urgency for this patient has been assessed as {nlp_result["urgency"]}, '
        f'which warrants appropriate and timely clinical response. '
        f'{action}'
    )

print('✅ Improved prompt and generation ready')

✅ Improved prompt and generation ready


In [ ]:
# ── Cell 6 FINAL: generate_report() — Flan-T5 only ───────────────
def generate_report(name, age, risk, xray, nlp_result):
    prompt = build_prompt(name, age, risk, xray, nlp_result)
    try:
        report = call_flan(prompt)
        source = 'Flan-T5 (local GPU)'
    except Exception as e:
        print(f'Flan failed: {e}')
        report = rule_based(name, age, risk, xray, nlp_result)
        source = 'Rule-based fallback'
    print(f'✅ Report via: {source}')
    return {'report': report}

# ── Test ─────────────────────────────────────────────────────────
r1 = generate_report(
    name='Ramesh Kumar', age=55,
    risk={'label': 'HIGH',     'score': 0.78},
    xray={'label': 'PNEUMONIA','confidence': 0.91},
    nlp_result={'symptoms': ['chest pain', 'breathlessness'], 'urgency': 'CRITICAL'}
)
print(f'\nPatient 1:\n{r1["report"]}\n' + '-'*60)

r2 = generate_report(
    name='Sunita Devi', age=42,
    risk={'label': 'MODERATE', 'score': 0.50},
    xray={'label': 'NORMAL',   'confidence': 0.85},
    nlp_result={'symptoms': ['fever', 'vomiting'], 'urgency': 'MODERATE'}
)
print(f'\nPatient 2:\n{r2["report"]}\n' + '-'*60)

r3 = generate_report(
    name='Arjun Singh', age=28,
    risk={'label': 'LOW',      'score': 0.15},
    xray={'label': 'NORMAL',   'confidence': 0.92},
    nlp_result={'symptoms': ['cough'], 'urgency': 'MILD'}
)
print(f'\nPatient 3:\n{r3["report"]}\n' + '-'*60)

✅ Report via: Flan-T5 (local GPU)

Patient 1:
Ramesh Kumar, a 55-year old patient with chest pain and breathlessness has been brought to the emergency department. He is at high cardiovascular risk for complications such as heart attack or stroke which requires immediate medical attention in order not further jeopardize his life expectancy of 6 months from now on! The overall triage urgency was CRITICICATIVE so urgent care should be provided right away because this condition could have potentially fatal consequences
------------------------------------------------------------
✅ Report via: Flan-T5 (local GPU)

Patient 2:
Sunita Devi, 42 years of age has been brought to the emergency department with fever and vomiting. The patient's overall triage urgency is MODERATE which means they are seriously afflicted but do not require immediate medical attention because their condition remains stable at this point in time; however there may still be complications that will need further investigat

In [ ]:
# ── Re-run generate_report() with improved functions ─────────────
r1 = generate_report(
    name='Ramesh Kumar', age=55,
    risk={'label': 'HIGH',     'score': 0.78},
    xray={'label': 'PNEUMONIA','confidence': 0.91},
    nlp_result={'symptoms': ['chest pain', 'breathlessness'], 'urgency': 'CRITICAL'}
)
print(f'\nPatient 1:\n{r1["report"]}\n' + '-'*60)

r2 = generate_report(
    name='Sunita Devi', age=42,
    risk={'label': 'MODERATE', 'score': 0.50},
    xray={'label': 'NORMAL',   'confidence': 0.85},
    nlp_result={'symptoms': ['fever', 'vomiting'], 'urgency': 'MODERATE'}
)
print(f'\nPatient 2:\n{r2["report"]}\n' + '-'*60)

r3 = generate_report(
    name='Arjun Singh', age=28,
    risk={'label': 'LOW',      'score': 0.15},
    xray={'label': 'NORMAL',   'confidence': 0.92},
    nlp_result={'symptoms': ['cough'], 'urgency': 'MILD'}
)
print(f'\nPatient 3:\n{r3["report"]}\n' + '-'*60)

✅ Report via: Flan-T5 (local GPU)

Patient 1:
Ramesh Kumar, a 55-year old patient with chest pain and breathlessness has been brought to the emergency department. He is at high cardiovascular risk for complications such as heart attack or stroke which requires immediate medical attention in order not further jeopardize his life expectancy of 6 months from now on! The overall triage urgency was CRITICICATIVE so urgent care should be provided right away because this condition could have potentially fatal consequences
------------------------------------------------------------
✅ Report via: Flan-T5 (local GPU)

Patient 2:
Sunita Devi, 42 years of age has been brought to the emergency department with fever and vomiting. The patient's overall triage urgency is MODERATE which means they are seriously afflicted but do not require immediate medical attention because their condition remains stable at this point in time; however there may still be complications that will need further investigat

In [ ]:
# ── Cell 7: Save llm_module.py to Drive ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Save the module file
llm_code = '''# Day 4 — LLM Module
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
_mdl = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(_dev)
_mdl.eval()

def _build_prompt(name, age, risk, xray, nlp_result):
    syms = ", ".join(nlp_result["symptoms"]) if nlp_result["symptoms"] else "none reported"
    urgency_desc = {
        "CRITICAL": "life-threatening, requires immediate intervention",
        "MODERATE": "serious but stable, requires prompt attention",
        "MILD":     "non-urgent, can be managed with routine care"
    }.get(nlp_result["urgency"], nlp_result["urgency"])
    return (
        f"You are a senior emergency medicine physician writing a formal clinical triage report. "
        f"A patient named {name}, {age} years old, has been brought to the emergency department. "
        f"The AI-based risk assessment has classified the cardiovascular risk as {risk[\\'label\\']} "
        f"with a probability score of {risk[\\'score\\']:.0%}. "
        f"The chest X-ray analysis returned a finding of {xray[\\'label\\']} "
        f"with a confidence of {xray[\\'confidence\\']:.0%}. "
        f"The patient reported the following symptoms: {syms}. "
        f"The overall triage urgency is {nlp_result[\\'urgency\\']}, meaning {urgency_desc}. "
        f"Write a detailed 5-sentence medical summary covering: "
        f"1) Patient and complaints. 2) AI risk and X-ray findings. "
        f"3) Clinical interpretation. 4) Urgency meaning. 5) Recommended action."
    )

def _rule_based(name, age, risk, xray, nlp_result):
    syms = ", ".join(nlp_result["symptoms"]) if nlp_result["symptoms"] else "none reported"
    action = {
        "CRITICAL": (
            "This patient must be transferred to the emergency resuscitation bay immediately. "
            "IV access, cardiac monitoring, and oxygen supplementation should be initiated without delay. "
            "A senior physician and crash team must be alerted at once."
        ),
        "MODERATE": (
            "The patient should be admitted to a monitored ward for further investigation. "
            "Blood investigations, ECG, and specialist consultation are recommended. "
            "Regular vitals monitoring every 2 hours is advised until stable."
        ),
        "MILD": (
            "The patient may be discharged with appropriate medications and advice. "
            "A follow-up with a general physician within 48 hours is recommended. "
            "Patient should return immediately if symptoms worsen."
        )
    }.get(nlp_result["urgency"], "Clinical evaluation by a physician is recommended.")
    risk_explain = {
        "HIGH":     "indicating significant likelihood of underlying cardiac pathology",
        "MODERATE": "suggesting moderate probability of cardiovascular involvement",
        "LOW":      "suggesting low probability of acute cardiac disease"
    }.get(risk["label"], "")
    xray_explain = {
        "PNEUMONIA": "consistent with active pulmonary infection requiring treatment",
        "NORMAL":    "showing no acute cardiopulmonary abnormality at this time"
    }.get(xray["label"], "")
    return (
        f"Patient {name}, {age} years of age, has presented to the emergency department "
        f"with the following reported symptoms: {syms}. "
        f"The AI risk assessment classified cardiovascular risk as {risk[\\'label\\']} "
        f"({risk[\\'score\\']:.0%} probability), {risk_explain}. "
        f"Chest X-ray returned {xray[\\'label\\']} ({xray[\\'confidence\\']:.0%} confidence), {xray_explain}. "
        f"Overall triage urgency is {nlp_result[\\'urgency\\']}, warranting timely clinical response. "
        f"{action}"
    )

def generate_report(name, age, risk, xray, nlp_result):
    prompt = _build_prompt(name, age, risk, xray, nlp_result)
    try:
        inp = _tok(prompt, return_tensors="pt", truncation=True, max_length=600).to(_dev)
        with torch.no_grad():
            out = _mdl.generate(
                **inp,
                max_new_tokens=300,
                min_new_tokens=80,
                num_beams=5,
                length_penalty=2.0,
                early_stopping=True,
                no_repeat_ngram_size=3,
                repetition_penalty=1.3
            )
        report = _tok.decode(out[0], skip_special_tokens=True).strip()
        if len(report.split()) < 30:
            raise ValueError("Output too short, using fallback")
        return {"report": report}
    except Exception as e:
        print(f"Flan fallback triggered: {e}")
        return {"report": _rule_based(name, age, risk, xray, nlp_result)}
'''

with open('/content/drive/MyDrive/fdp/llm_module.py', 'w') as f:
    f.write(llm_code)

print('✅ llm_module.py saved to Drive')

Mounted at /content/drive
✅ llm_module.py saved to Drive


In [ ]:
# ── Cell 8: Verify file saved correctly ──────────────────────────
import os

path = '/content/drive/MyDrive/fdp/'
files = os.listdir(path)
print('📁 Files in /fdp/ folder:')
for f in sorted(files):
    size = os.path.getsize(path + f)
    print(f'   {f:30s}  {size/1024:.1f} KB')

📁 Files in /fdp/ folder:
   llm_module.py                   4.7 KB
   nlp_model                       4.0 KB
   nlp_module.py                   1.3 KB


In [ ]:
# ── Cell 10: Save Flan-T5 model to Drive ─────────────────────────
import os

save_path = '/content/drive/MyDrive/fdp/llm_model'
os.makedirs(save_path, exist_ok=True)

print('Saving Flan-T5 tokenizer...')
flan_tok.save_pretrained(save_path)

print('Saving Flan-T5 model...')
flan_mdl.save_pretrained(save_path)

print('✅ Flan-T5 model saved to Drive')

Saving Flan-T5 tokenizer...
Saving Flan-T5 model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Flan-T5 model saved to Drive


In [ ]:
# ── Cell 11: Verify model files saved ────────────────────────────
import os

save_path = '/content/drive/MyDrive/fdp/llm_model'
files = os.listdir(save_path)

print('📁 Files in /fdp/llm_model/:')
total = 0
for f in sorted(files):
    size = os.path.getsize(os.path.join(save_path, f))
    total += size
    print(f'   {f:40s}  {size/1024/1024:.1f} MB')
print(f'\n   Total size: {total/1024/1024:.1f} MB')

📁 Files in /fdp/llm_model/:
   config.json                               0.0 MB
   generation_config.json                    0.0 MB
   model.safetensors                         2987.5 MB
   tokenizer.json                            2.0 MB
   tokenizer_config.json                     0.0 MB

   Total size: 2989.6 MB


In [ ]:
# ── Cell 12: Update llm_module.py to load from Drive ─────────────
llm_code_updated = '''# Day 4 — LLM Module | Anand Kumar
# Loads Flan-T5 from saved Drive path (faster than downloading each time)
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "/content/drive/MyDrive/fdp/llm_model"   # local Drive path
_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading Flan-T5 from Drive on {_dev}...")
_tok = AutoTokenizer.from_pretrained(MODEL_PATH)
_mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(_dev)
_mdl.eval()
print("✅ Flan-T5 loaded from Drive")

def _build_prompt(name, age, risk, xray, nlp_result):
    syms = ", ".join(nlp_result["symptoms"]) if nlp_result["symptoms"] else "none reported"
    urgency_desc = {
        "CRITICAL": "life-threatening, requires immediate intervention",
        "MODERATE": "serious but stable, requires prompt attention",
        "MILD":     "non-urgent, can be managed with routine care"
    }.get(nlp_result["urgency"], nlp_result["urgency"])
    return (
        f"You are a senior emergency medicine physician writing a formal clinical triage report. "
        f"A patient named {name}, {age} years old, has been brought to the emergency department. "
        f"The AI-based risk assessment has classified the cardiovascular risk as {risk[\\'label\\']} "
        f"with a probability score of {risk[\\'score\\']:.0%}. "
        f"The chest X-ray analysis returned a finding of {xray[\\'label\\']} "
        f"with a confidence of {xray[\\'confidence\\']:.0%}. "
        f"The patient reported the following symptoms: {syms}. "
        f"The overall triage urgency is {nlp_result[\\'urgency\\']}, meaning {urgency_desc}. "
        f"Write a detailed 5-sentence medical summary covering: "
        f"1) Patient and complaints. 2) AI risk and X-ray findings. "
        f"3) Clinical interpretation. 4) Urgency meaning. 5) Recommended action."
    )

def _rule_based(name, age, risk, xray, nlp_result):
    syms = ", ".join(nlp_result["symptoms"]) if nlp_result["symptoms"] else "none reported"
    action = {
        "CRITICAL": (
            "This patient must be transferred to the emergency resuscitation bay immediately. "
            "IV access, cardiac monitoring, and oxygen supplementation should be initiated without delay. "
            "A senior physician and crash team must be alerted at once."
        ),
        "MODERATE": (
            "The patient should be admitted to a monitored ward for further investigation. "
            "Blood investigations, ECG, and specialist consultation are recommended. "
            "Regular vitals monitoring every 2 hours is advised until stable."
        ),
        "MILD": (
            "The patient may be discharged with appropriate medications and advice. "
            "A follow-up with a general physician within 48 hours is recommended. "
            "Patient should return immediately if symptoms worsen."
        )
    }.get(nlp_result["urgency"], "Clinical evaluation by a physician is recommended.")
    risk_explain = {
        "HIGH":     "indicating significant likelihood of underlying cardiac pathology",
        "MODERATE": "suggesting moderate probability of cardiovascular involvement",
        "LOW":      "suggesting low probability of acute cardiac disease"
    }.get(risk["label"], "")
    xray_explain = {
        "PNEUMONIA": "consistent with active pulmonary infection requiring treatment",
        "NORMAL":    "showing no acute cardiopulmonary abnormality at this time"
    }.get(xray["label"], "")
    return (
        f"Patient {name}, {age} years of age, has presented to the emergency department "
        f"with the following reported symptoms: {syms}. "
        f"The AI risk assessment classified cardiovascular risk as {risk[\\'label\\']} "
        f"({risk[\\'score\\']:.0%} probability), {risk_explain}. "
        f"Chest X-ray returned {xray[\\'label\\']} ({xray[\\'confidence\\']:.0%} confidence), {xray_explain}. "
        f"Overall triage urgency is {nlp_result[\\'urgency\\']}, warranting timely clinical response. "
        f"{action}"
    )

def generate_report(name, age, risk, xray, nlp_result):
    prompt = _build_prompt(name, age, risk, xray, nlp_result)
    try:
        inp = _tok(prompt, return_tensors="pt", truncation=True, max_length=600).to(_dev)
        with torch.no_grad():
            out = _mdl.generate(
                **inp,
                max_new_tokens=300,
                min_new_tokens=80,
                num_beams=5,
                length_penalty=2.0,
                early_stopping=True,
                no_repeat_ngram_size=3,
                repetition_penalty=1.3
            )
        report = _tok.decode(out[0], skip_special_tokens=True).strip()
        if len(report.split()) < 30:
            raise ValueError("Output too short, using fallback")
        return {"report": report}
    except Exception as e:
        print(f"Flan fallback triggered: {e}")
        return {"report": _rule_based(name, age, risk, xray, nlp_result)}
'''

with open('/content/drive/MyDrive/fdp/llm_module.py', 'w') as f:
    f.write(llm_code_updated)

print('✅ llm_module.py updated to load model from Drive')

✅ llm_module.py updated to load model from Drive


In [ ]:
# ── Cell 13: Final Drive folder summary ──────────────────────────
import os

base = '/content/drive/MyDrive/fdp/'
print('📁 Complete /fdp/ folder — Day 4 final state:\n')

for item in sorted(os.listdir(base)):
    full = os.path.join(base, item)
    if os.path.isdir(full):
        sub_files = os.listdir(full)
        sub_size  = sum(os.path.getsize(os.path.join(full, f)) for f in sub_files)
        print(f'   📂 {item:35s}  {sub_size/1024/1024:.1f} MB  ({len(sub_files)} files)')
    else:
        size = os.path.getsize(full)
        print(f'   📄 {item:35s}  {size/1024:.1f} KB')

📁 Complete /fdp/ folder — Day 4 final state:

   📂 llm_model                            2989.6 MB  (5 files)
   📄 llm_module.py                        4.9 KB
   📂 nlp_model                            256.1 MB  (5 files)
   📄 nlp_module.py                        1.3 KB
